# `6.steane` — Steane [[7,1,3]] error correction

This notebook is the MLIR-stack counterpart of `examples/6.steane.ipynb`.
The compiler expands every logical qubit to seven physical qubits, prepares
encoded `|0>`, applies supported Cliffords transversally, performs explicit
syndrome extraction and correction, and decodes seven measurements back to
one logical bit.

In [ ]:
%load_ext qstack_mlir.jupyter

import logging

logger = logging.getLogger("qstack")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(handler)

from qstack_mlir.passes.steane import compile_steane, register_steane_callbacks
from qstack_mlir.runtime import CallbackRegistry, Machine

## Encoded logical one

In [ ]:
%%qasm logical_one
QSTACKQASM 0.1;
include "qstack/cliffords.inc";

qreg q[1];
creg c[1];
x q[0];
measure q[0] -> c[0];

In [ ]:
encoded_one = compile_steane(logical_one)
registry = CallbackRegistry()
register_steane_callbacks(registry)

# Inspect the widened allocation, transversal X, syndrome rounds, and decoder.
print(encoded_one)

Machine(encoded_one, num_qubits=10, registry=registry).shots("main", 20).histogram()

The seven data wires remain live while each syndrome round temporarily uses
three ancillas, so a single logical qubit needs a ten-wire emulator budget.

## Encoded Bell state

In [ ]:
%%qasm logical_bell
QSTACKQASM 0.1;
include "qstack/cliffords.inc";

qreg q[2];
creg c[2];
h q[0];
cx q[0], q[1];
measure q[0] -> c[0];
measure q[1] -> c[1];

In [ ]:
encoded_bell = compile_steane(logical_bell)
print(encoded_bell)

In [ ]:
# Trace one evaluation through physical gates, syndrome selection, and decoding.
logger.setLevel(logging.DEBUG)
machine = Machine(encoded_bell, num_qubits=17, registry=registry)
machine.shots("main", 1).data

In [ ]:
logger.setLevel(logging.INFO)
machine.shots("main", 40).plot_histogram()

Only `(0, 0)` and `(1, 1)` are reachable: the Steane transformation preserves
the logical Bell correlation.

## Compose Steane with H2 lowering

In [ ]:
from qstack_mlir.passes.cliffords2h2 import compile_cliffords_to_h2

compile_cliffords_to_h2(encoded_one)
Machine(encoded_one, num_qubits=10, registry=registry).shots("main", 5).histogram()

The QEC pass and hardware lowering compose directly: generated preparation,
syndrome, and correction Cliffords all lower to H2-native operations.